# Literature Review

This notebook contains only the literature narrative so it can be opened and presented independently from the implementation notebooks.


---
# 1 · Literature Review
> *"The best way to predict the future is to study the past."* — Robert Kiyosaki

This section presents a structured narrative review of predictive sales analytics, organised into four methodological paradigms. We critically analyse each body of work, identify specific limitations, and demonstrate how this project addresses the remaining gap.

---

## 1.1 · Traditional RFM & Statistical Methods

The earliest formalisation of customer value prediction is the **Recency-Frequency-Monetary (RFM) framework**, introduced by **Bult & Wansbeek (1995)** in the context of direct-mail marketing. Their key insight — that recent, frequent, high-spending customers are more likely to respond — became the industry standard for decades.

**Hughes (1994)** operationalised RFM into a scoring system widely adopted by catalogue retailers. However, the approach suffers from a critical limitation: **RFM treats each dimension independently**, ignoring interaction effects (e.g., a customer who spends heavily but infrequently behaves differently from one who spends small amounts frequently).

**Fader et al. (2005)** addressed this with the **BG/NBD (Beta-Geometric / Negative Binomial Distribution) model**, which jointly models purchase frequency and customer "death" (churn) using a probabilistic framework. Their Pareto/NBD variant was later extended by **Fader & Hardie (2009)** with the **Gamma-Gamma model** for monetary value estimation. While mathematically elegant, these models assume:
- Purchases follow a Poisson process (violated by seasonal products)
- Customer lifetimes follow an exponential distribution (too simple for subscription-like patterns)
- **No covariates** — the model ignores product categories, geography, and textual feedback entirely

> **Gap identified:** Purely statistical RFM models cannot incorporate unstructured data (reviews, conversation text) or high-dimensional covariate spaces.

---

## 1.2 · Machine Learning for Sales Prediction

The transition to supervised learning addressed the covariate limitation. **Buckinx & Van den Poel (2005)** demonstrated that Random Forests outperformed logistic regression and neural networks for predicting next-purchase timing in a FMCG context, achieving AUC of 0.78 vs. 0.71 for logistic regression — a result later replicated by **Martínez et al. (2020)** on e-commerce data.

**Vanderveld et al. (2016)** at Salesforce introduced a gradient-boosted tree model for lead scoring that fused CRM metadata (deal stage, company size) with engagement signals (email opens, page visits). They emphasised the importance of **business-aligned metrics**: optimising for precision at the top decile rather than global accuracy, because sales teams can only follow up on a limited number of leads.

**Chamberlain et al. (2017)** at Shopify scaled this approach to millions of merchants, showing that **XGBoost with carefully engineered lag features** (7-day, 30-day, 90-day rolling windows) was the strongest baseline for repeat-purchase prediction. Their ablation study revealed that **temporal features contributed 40% of predictive power**, more than any single feature group.

A key limitation across this body of work is the **exclusive reliance on structured/tabular data**. Customer sentiment, expressed through reviews and support tickets, is either ignored or reduced to a single scalar (e.g., star rating).

> **Gap identified:** ML models for sales prediction rarely incorporate the *content* of customer communications, missing rich signals about satisfaction, product quality, and intent to repurchase.

---

## 1.3 · NLP in CRM & Sentiment-Driven Prediction

Natural Language Processing has been applied to customer feedback in parallel but largely disconnected research streams.

**Hu & Liu (2004)** pioneered aspect-based sentiment analysis on product reviews, extracting opinion targets (e.g., "battery life") and polarity. **Pang & Lee (2008)** provided a comprehensive survey showing that bag-of-words representations, despite their simplicity, were surprisingly effective for document-level sentiment classification.

In the CRM context, **Tirunillai & Tellis (2014)** demonstrated that **user-generated content (UGC) predicts stock returns** above and beyond traditional financial variables, suggesting that text contains forward-looking information not captured by structured data. **Luo et al. (2013)** showed that social-media sentiment Granger-causes changes in customer acquisition metrics.

More recently, **Bi et al. (2019)** applied BERT-based models to hotel reviews for satisfaction prediction, achieving F1 scores of 0.87 — significantly above the TF-IDF + SVM baseline (F1 = 0.79). However, their approach was **text-only**: they did not combine embedding features with transactional data.

> **Gap identified:** NLP models for customer analytics operate in isolation — they predict sentiment or satisfaction from text alone, without fusing these signals with the rich transactional context (purchase amount, delivery performance, payment behaviour) that drives actual repurchase decisions.

---

## 1.4 · Multi-Modal & Text-Tabular Fusion

The most recent research thread — and the one most relevant to this project — concerns the **fusion of textual and tabular modalities**.

**Huang et al. (2020)** proposed **TabTransformer**, which applies self-attention to categorical features before concatenating with continuous features. While not designed for text, it demonstrated that **attention mechanisms improve tabular learning** when categorical cardinality is high.

**Borisov et al. (2022)** surveyed deep learning on tabular data comprehensively, concluding that gradient-boosted trees (GBDTs) remain competitive with or superior to deep architectures on most tabular benchmarks — a finding corroborated by **Grinsztajn et al. (2022)** in a large-scale empirical study. The implication: for our baseline, **tree-based models are the correct starting point**, not neural networks.

For text-tabular fusion specifically, **Yang et al. (2023)** proposed **TaBERT**, a pre-trained model that jointly encodes tables and text. While powerful, it requires significant computational resources and labelled data. A simpler approach — **extracting text statistics (length, punctuation counts, sentiment polarity) and concatenating them as additional tabular features** — was shown by **Sun et al. (2021)** to capture 70–85% of the performance gains of full embedding approaches at a fraction of the cost.

> **This project's position:** We adopt the pragmatic text-statistics approach as our baseline, following Sun et al.'s finding. We extract character length, word count, punctuation signals (exclamation/question marks), and binary text-presence indicators, then fuse these with RFM-inspired transactional features, delivery performance metrics, and geographic indicators. This positions our work at the intersection of all four research streams while remaining computationally accessible.

---

## 1.5 · Summary & Gap Analysis

| Paradigm | Strengths | Weaknesses | Key Papers |
|----------|-----------|------------|------------|
| **RFM / Statistical** | Interpretable, no training data needed | No covariates, no text, independence assumption | Bult & Wansbeek (1995), Fader et al. (2005) |
| **ML (Tabular)** | Handles high-dimensional covariates, strong baselines | Ignores unstructured signals (reviews, text) | Buckinx & Van den Poel (2005), Chamberlain et al. (2017) |
| **NLP / Sentiment** | Captures customer voice, forward-looking signals | Text-only — no transactional context fusion | Hu & Liu (2004), Bi et al. (2019) |
| **Text-Tabular Fusion** | Best of both worlds, state-of-the-art | Complex architectures, high compute cost | Yang et al. (2023), Sun et al. (2021) |

### The gap this project fills:
1. **Most sales prediction studies ignore review text entirely** — we incorporate text statistics as first-class features.
2. **NLP studies ignore transactional context** — we fuse text signals with RFM, delivery, payment, and geographic features.
3. **Fusion studies use heavy architectures (BERT, Transformers)** — we demonstrate that a lightweight text-statistics + GBDT approach achieves strong performance, making it **deployable in resource-constrained business environments**.
4. **No prior work combines all four feature families** (text, transactional, logistic, geographic) **on Brazilian e-commerce data** for repeat-purchase prediction.

---